# 新版机器学习滚动训练模版


通过AI方法构建的量化选股与动态调整

In [6]:
from bigquant import bigtrader, dai
import pandas as pd
from datetime import datetime, timedelta
import numpy as np
from sklearn.linear_model import LinearRegression
import talib
from sklearn.model_selection import train_test_split
import lightgbm as lgb
from lightgbm import early_stopping
from sklearn.linear_model import LinearRegression


def initialize(context: bigtrader.IContext):
    from bigtrader.finance.commission import PerOrder

    context.holding_days = 5
    context.target_hold_count = 10
    context.train_days = 120
    context.retrain = 10

    context.pred = []
    context.sv = 0
    # 系统已经设置了默认的交易手续费和滑点，要修改手续费可使用如下函数
    context.set_commission(PerOrder(buy_cost=0.0003, sell_cost=0.0013, min_cost=5))
    context.sql = f"""
SELECT
    p.date,
    p.instrument,
    p.close,
    p.open,
    (LAG(p.close,-{context.holding_days}/p.close) OVER (PARTITION BY p.instrument ORDER BY p.date)) AS holding_return,
    c_pct_rank(p.turn) as turn,
    c_pct_rank(p.volume) as volume,
    c_pct_rank(p.amount) as amount,
    c_pct_rank(p.obv) as obv,
    c_pct_rank(p.pe_ttm) as pe_ttm,
    c_pct_rank(p.float_market_cap) as float_market_cap,
    p.bias_5,
    p.ema_5,
    0.01 * ta_ema(p.amount, 13) OVER (PARTITION BY p.instrument ORDER BY p.date) / ta_ema(p.volume, 13) OVER (PARTITION BY p.instrument ORDER BY p.date) AS x3,
    -- (p.close-x3)/x3*1000 as ZZ,
    -- p.kdj_d_9_3_3,
    -- p.kdj_k_9_3_3,
    -- p.kdj_j_9_3_3,
    -- p.kdj_rsv_9_3_3,
    p.rsi_12,
FROM cn_stock_prefactors p
WHERE
    p.list_sector = 1 AND
    p.st_status = 0 AND
    pe_ttm > 0 and
    list_days > 365
    """

    # 1. 换手率
    # 1. 涨跌幅
    # 2. 成交量
    # 3. 成交额
    # 3. 平均成交量
    # 3. 乖离率
    # 3. 概念与行业
    # 4. 热度排名
    # 5. 技术指标
    #   1. MACD
    #   2. RSI
    #   3. 布林带
    #   4. 均线



def befor_trading(context, data):
    context.sv = context.portfolio.portfolio_value
    current_date = data.current_dt.strftime("%Y-%m-%d")
    date_str = current_date
    print(current_date, context.sv)

    # TODO
    # =========定义训练列========
    # HINT  因子列名
    # 此处默认使用全部列
    # =========================
    x_train = pd.DataFrame()
    x_val = pd.DataFrame()
    y_train = pd.DataFrame()
    y_val = pd.DataFrame()
    train = pd.DataFrame()
    test = pd.DataFrame()
    x_test = pd.DataFrame()
    if context.trading_day_index % context.retrain != 0:
        try:
            date_obj = datetime.strptime(date_str, "%Y-%m-%d")
            n_days_ago = date_obj - timedelta(days=context.train_days)
            df = dai.query(
                context.sql, filters={"date": [n_days_ago, current_date]}
            ).df()
            all_data = df.sort_values(
                ["instrument", "date"], ascending=[True, True]
            ).dropna()
            # ============机器学习训练模型==========

            # (v, s, t) = talib.MACD(all_data['close'].to_numpy())
            # all_data["market_macd_v"] = v
            # all_data["market_macd_s"] = s
            # all_data["market_macd_t"] = t
            # HINT 对因子和label进行划分
            all_data["label"] = all_data["holding_return"]
            train_col = all_data.columns.difference(
                ["date", "instrument", "holding_return", "label"]
            )
            # 训练模型
            train = all_data[all_data.date < current_date]
            test = all_data[current_date == all_data.date]
            x = train[train_col]
            y = train["label"]
            # display(test_date)
            x_train, x_val, y_train, y_val = train_test_split(
                x, y, test_size=0.2, random_state=42
            )
            x_test = test[train_col]
            # ==================================
            # display(test)
        except Exception as e:
            print(e)
    # return
    if context.trading_day_index % context.holding_days != 0:

        # TODO
        # ============模型预测==================
        # 5. 模型训练
        # model = lgb.LGBMRegressor(
        #     n_estimators=200,
        #     learning_rate=0.1,
        #     num_leaves=36,  # 添加叶子节点限制
        #     max_depth=-1,  # 设置合理深度限制
        #     subsample=0.85,
        #     colsample_bytree=0.85,  # 添加特征采样
        #     random_state=72,
        #     force_col_wise=True,
        #     verbose=-1,
        # )

        # model.fit(
        #     x_train,
        #     y_train,
        #     eval_set=[(x_val, y_val)],
        #     eval_metric="l2",
        #     callbacks=[early_stopping(stopping_rounds=50, verbose=True)],
        # )
        # # 因子影响力方法: 按信息增益计算 (通常更准确)
        # gain_importance = pd.Series(
        #     model.booster_.feature_importance(importance_type="gain"),
        #     index=x_train.columns,
        # ).sort_values(ascending=False)
        # print("\n按信息增益的重要性:")
        # print(gain_importance)
        # 创建模型
        model = LinearRegression()
        model.fit(
            x_train,
            y_train,
        )

        # 6. 预测并筛选当天结果
        test["score"] = model.predict(x_test)
        # =====================================
        context.pred = test
        # display(test)


def handle_data(context: bigtrader.IContext, data: bigtrader.IBarData):
    # return
    import pandas as pd
    from bigtrader.constant import Direction
    from bigtrader.constant import OrderType

    if len(context.pred) < 1:
        return

    df = context.pred

    buy_list = list(
        df.sort_values(by="score", ascending=False)
        .head(context.target_hold_count)
        .instrument
    )
    len_ = len(buy_list)
    target_hold_instruments = set(buy_list)

    # 获取当前已持有股票
    current_hold_instruments = set(context.get_account_positions().keys())

    # 卖出不在目标持有列表中的股票
    for instrument in current_hold_instruments - target_hold_instruments:
        context.order_target_percent(instrument, 0)

    # 买入目标持有列表中的股票
    for instrument in target_hold_instruments - current_hold_instruments:
        context.order_target_percent(instrument, 1 / len_)

    context.pred = []


performance = bigtrader.run(
    market=bigtrader.Market.CN_STOCK,
    frequency=bigtrader.Frequency.DAILY,
    start_date="2024-01-01",
    end_date="2025-07-10",
    capital_base=3000000,  # 设置初始资金
    initialize=initialize,  # 传入初始化函数
    handle_data=handle_data,  # 传入数据处理函数
    before_trading_start=befor_trading,
    order_price_field_buy="open",
    order_price_field_sell="close",
)

# 渲染绩效报告，展示回测结果
performance.render()

[2025-08-11 22:37:39] [info     ] bigtrader init ..
[2025-08-11 22:37:39] [info     ] bigtrader.run start: market=cn_stock, frequency=1d, mode=backtest, start_date=2024-01-01, end_date=2025-07-10
[2025-08-11 22:37:39] [info     ] bigtrader<backtest> init ..
[2025-08-11 22:37:39] [info     ] prepare data ..
[2025-08-11 22:37:44] [info     ] bar1d_df: (2195669, 16)
[2025-08-11 22:37:44] [info     ] bigtrader run ..
2024-01-02 3000000.0
2024-01-03 3000000.0
2024-01-04 3000000.0
2024-01-05 2988691.2100000004
2024-01-08 2977287.2100000004
2024-01-09 2965127.2100000004
2024-01-10 2943529.2100000004
2024-01-11 2917850.2100000004
2024-01-12 2950022.2100000004
2024-01-15 2923390.2100000004
2024-01-16 2922202.2100000004
2024-01-17 2947310.2100000004
2024-01-18 2887831.2100000004
2024-01-19 2917712.2100000004
2024-01-22 2908199.2100000004
2024-01-23 2818341.2100000004
2024-01-24 2820538.2100000004
2024-01-25 2892813.2100000004
2024-01-26 2993480.2100000004
2024-01-29 3027455.2100000004
2024-01-30

    file:pystrategy.cpp line:169 desc:before_trading


ValueError: at least one array or dtype is required

TradeEngine cached backtest exception: invoke_strategy_method: exception at pystrategy.cpp:169:before_trading



Exception: trade_engine.run error -18: StrategyException

In [ ]:
from IPython import get_ipython
import nbformat
import os
current_cell = get_ipython().get_parent()
assert current_cell["metadata"]["cellId"].startswith("vscode-notebook-cell://bigquant.com")
path = current_cell["metadata"]["cellId"].replace("vscode-notebook-cell://bigquant.com", "").split("#")[0]


path = "/home/aiuser/work/AIQuANT/策略复刻/AI天创.ipynb"
notebook = nbformat.read(path, as_version=4)

text = notebook.cells[0].source.strip()
if not text.startswith("#"):
    raise Exception("invalid notebook")


import hashlib
name = os.path.splitext(os.path.basename(path))[0] 
_id = hashlib.md5(name.encode()).hexdigest()
print(name, _id )


name, desc = text.split("\n", 1)
name = name[1:].strip()
desc = desc.strip()
code = notebook.cells[1].source.strip()

algorithm_period_return = performance.raw_perf["algorithm_period_return"]
algorithm_period_return_list = [[date.strftime('%Y-%m-%d'), value] for date, value in zip(algorithm_period_return.index, algorithm_period_return.values)]

from bigtrader._v2.performancechartrender import PerformanceChartRender

perf_summary = PerformanceChartRender(
    raw_perf=performance.raw_perf,
    account_raw_performances=performance.account_raw_performances,
    market=performance.market,
    frequency=performance.frequency,
    logger=None,
    bar1d=None
).eval_perf_summary()


def _build_items(perf_df, col, max_rows):
    items = []
    for i in range(len(perf_df[col])):
        day_postions = perf_df[col].iloc[-i - 1]
        date = perf_df.index[-i - 1].strftime('%Y-%m-%d')
        for i in range(min(max_rows, len(day_postions))):
            day_postions[i]["date"] = date

        items += day_postions
        if len(items) >= max_rows:
            break
    return items[:10]

strategy = {
   # "space_id": "00000000-0000-0000-0000-000000000000",
    "id": _id,
    "strategy_name": name,
    "strategy_desc": desc,
    "strategy_content": "",
    "meta_data": {},
    "market_tag":'stock',
    
    "strategy_code": code,
    "summary_data": {
        "net_values": algorithm_period_return_list,
        "perf_summary": perf_summary,
    },
    "data": {
        "positions": _build_items(performance.raw_perf, "positions", 10),
        "transactions": _build_items(performance.raw_perf, "transactions", 10),
    },
}
# performance.raw_perf
# strategy["code"]

import httpx
response = httpx.post("http://tradingapiserver.bigquant:8000/bigapis/trading/v1/community/aistrategy", json=strategy).json()

print(f"https://bigquant.com/square/ai/{response['data']['id']}")

FileNotFoundError: [Errno 2] No such file or directory: '/home/aiuser/work/AIQuANT/策略复刻/AI天创.ipynb'

In [ ]:
desc